# Aula 09 - Notebook: Motores de Inferência Forward e Backward Chaining

Neste notebook implementamos algoritmos completos de **Forward Chaining** e **Backward Chaining** para diagnóstico automatizado em tempo de execução.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass
from typing import Set, Tuple, List, Dict, Optional, Any

@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1

class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []
        
    def adicionar_regra(self, id_r: str, antecedentes: List[str], consequente: str, desc: str, prioridade: int = 1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))

class MotorInferencia:
    def __init__(self, base_conhecimento: BaseConhecimento):
        self.bc = base_conhecimento
        
    def forward_chaining(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        fatos_conhecidos = set(fatos_iniciais)
        historico_disparos = []
        novos_fatos = True
        passo = 1
        
        while novos_fatos:
            novos_fatos = False
            regras_candidatas = sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True)
            for regra in regras_candidatas:
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico_disparos.append({
                        "Passo": passo,
                        "Regra": regra.id_regra,
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico": regra.descricao_diagnostico
                    })
                    passo += 1
                    novos_fatos = True
                    break
        return fatos_conhecidos, historico_disparos

bc = BaseConhecimento()
bc.adicionar_regra("R-01", ["p1", "t1"], "reacao_runaway", "Exotermia Descontrolada", 10)
bc.adicionar_regra("R-02", ["reacao_runaway", "v1"], "trip_nh3", "Fechamento Imediato Válvula NH3", 10)

motor = MotorInferencia(bc)
fatos_finais, trilha = motor.forward_chaining({"p1", "t1", "v1"})
print("Trilha de Diagnóstico Forward Chaining:")
print(formatar_tabela(trilha))
assert "trip_nh3" in fatos_finais


Trilha de Diagnóstico Forward Chaining:
Passo | Regra | Fato Inferido  | Diagnóstico                    
------+-------+----------------+--------------------------------
1     | R-01  | reacao_runaway | Exotermia Descontrolada        
2     | R-02  | trip_nh3       | Fechamento Imediato Válvula NH3
